In [1]:
import torch
import numpy as np
import plotly.express as px

from kaolin.render.camera import Camera
from kaolin.render.mesh import dibr_rasterization
from kaolin.visualize import IpyTurntableVisualizer

from jaxtyping import Float32, Int32, UInt8, jaxtyped
from beartype import beartype

In [2]:
backend = 'nvdiffrast'
device = 'cuda'

height = 1080
width = 1920

In [3]:
sphere_knots_u = torch.tensor([0.0, 0.0, 0.0, 0.5, 0.5, 1.0, 1.0, 1.0], dtype=torch.float32, device=device)
sphere_knots_v = torch.tensor([0.0, 0.0, 0.0, 0.25, 0.25, 0.5, 0.5, 0.75, 0.75, 1.0, 1.0, 1.0], dtype=torch.float32, device=device)

sphere_control_points = torch.tensor([
    [0.0000, 0.0000, -1.0000, 1.0000],
    [-1.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, 0.0000, 0.0000, 1.0000],
    [-1.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, -1.0000, -1.0000, 0.5000],
    [-1.0000, -1.0000, 0.0000, 0.7071],
    [-1.0000, -1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [0.0000, -1.0000, -1.0000, 0.7071],
    [0.0000, -1.0000, 0.0000, 1.0000],
    [0.0000, -1.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, -1.0000, -1.0000, 0.5000],
    [1.0000, -1.0000, 0.0000, 0.7071],
    [1.0000, -1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [1.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, 0.0000, 0.0000, 1.0000],
    [1.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, 1.0000, -1.0000, 0.5000],
    [1.0000, 1.0000, 0.0000, 0.7071],
    [1.0000, 1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [0.0000, 1.0000, -1.0000, 0.7071],
    [0.0000, 1.0000, 0.0000, 1.0000],
    [0.0000, 1.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, 1.0000, -1.0000, 0.5000],
    [-1.0000, 1.0000, 0.0000, 0.7071],
    [-1.0000, 1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
], dtype=torch.float32, device=device).reshape(8, 5, 4)

sphere_control_points_blue = sphere_control_points.clone()
sphere_control_points_blue[..., 1] = sphere_control_points_blue[..., 1] + 2

color_red = torch.tensor([255, 0, 0], dtype=torch.float32, device=device)
color_blue = torch.tensor([0, 0, 255], dtype=torch.float32, device=device)


In [4]:
class NURBS:

    @jaxtyped(typechecker=beartype)
    def __init__(
            self,
            color: Float32[torch.Tensor, "3"],
            control_points: Float32[torch.Tensor, "V U 4"],
            vector_knot_u: Float32[torch.Tensor, "KnotU"],
            vector_knot_v: Float32[torch.Tensor, "KnotV"],
            degree: int = 3,
            cycle: bool = True,
            device: str = 'cuda'
    ):
        self.color = color.to(device)
        self.control_points = control_points.to(device)
        self.vector_knot_u = vector_knot_u.to(device)
        self.vector_knot_v = vector_knot_v.to(device)
        self.degree = degree
        self.cycle = cycle
        self.device = device


    @jaxtyped(typechecker=beartype)
    def cox_de_boor(
            self,
            discretisation: Float32[torch.Tensor, "Res"],
            knots: Float32[torch.Tensor, "Knot"]
    ) -> Float32[torch.Tensor, "Res Basis"]:

        def divide(
                numerator: torch.Tensor,
                denominator: torch.Tensor,
                thresh: float = 1e-6
        ) -> torch.Tensor:
            return torch.where(denominator.abs() < thresh, torch.zeros_like(numerator), numerator / denominator)

        discretisation = discretisation.unsqueeze(-1)
        lower_bound = knots[..., :-1]
        upper_bound = knots[..., 1:]

        epsilon = 1e-6
        max_val = knots.max() - epsilon
        clamped = torch.min(discretisation, max_val)

        b = ((clamped >= lower_bound) & (clamped < upper_bound)).float()

        for d in range(1, self.degree + 1):
            left = divide((discretisation - knots[..., :-(d + 1)]), (knots[..., d:-1] - knots[..., :-(d + 1)]))
            right = divide((knots[..., (d + 1):] - discretisation), (knots[..., (d + 1):] - knots[..., 1: -d]))

            b = left * b[..., :-1] + right * b[..., 1:]

        return b

    @jaxtyped(typechecker=beartype)
    def tessellate(self) -> tuple[Float32[torch.Tensor, "N 3"], Int32[torch.Tensor, "M 3"]]:

        weighted_control_points = torch.cat([self.control_points, self.control_points[ :self.degree - 1]])
        weighted_control_points[..., :3] = weighted_control_points[..., :3] * weighted_control_points[..., 3:4]

        resolution_u = weighted_control_points.shape[1] * 10
        resolution_v = weighted_control_points.shape[0] * 10

        b_u = self.cox_de_boor(torch.linspace(0, 1, resolution_u, device=self.device), self.vector_knot_u)
        b_v = self.cox_de_boor(torch.linspace(0, 1, resolution_v, device=self.device), self.vector_knot_v)

        coordinate_4d = torch.einsum('iu, jv, vuw -> ijw', b_u, b_v, weighted_control_points)

        coordinate_3d = coordinate_4d[..., :3] / coordinate_4d[..., 3:4]
        coordinate_3d = coordinate_3d.reshape(-1, 3)

        triangles = []

        def idx(
                u: int,
                v: int
        ) -> int:
            return u * resolution_v + v

        for u in range(resolution_u - 1):
            for v in range(resolution_v - 1):
                b_l = idx(u, v)
                b_r = idx(u, v + 1)
                t_l = idx(u + 1, v)
                t_r = idx(u + 1, v + 1)

                triangles.append([b_l, b_r, t_l])
                triangles.append([b_r, t_r, t_l])

        return coordinate_3d, torch.tensor(triangles, dtype=torch.int32, device=self.device)

In [5]:
sphere_red = NURBS(color_red, sphere_control_points, sphere_knots_u, sphere_knots_v, 2)
sphere_blue = NURBS(color_blue, sphere_control_points_blue, sphere_knots_u, sphere_knots_v, 2)

scene = [sphere_red, sphere_blue]

In [6]:
camera = Camera.from_args(
    eye=torch.tensor([10.0, 0.0, 0.0]),
    at=torch.tensor([1.0, 1.0, 0.0]),
    up=torch.tensor([0.0, 1.0, 0.0]),
    fov=30 * np.pi / 180,
    width=width, height=height,
    near=1e-2, far=1e2,
    dtype=torch.float32,
    device='cuda'
)

cameras = [camera, camera]

In [7]:
@jaxtyped(typechecker=beartype)
def render(
    scene: list[NURBS],
    cameras: list[Camera],
    H: int = 1080,
    W: int = 1920
): #-> Float32[torch.Tensor, "Cam H W 3"]:
    verts_vector = []
    faces_vector = []
    colors_vector = []
    offset = 0

    for nurbs in scene:
        verts_k, faces_k = nurbs.tessellate()

        verts_vector.append(verts_k)
        faces_vector.append(faces_k + offset)
        colors_vector.append(nurbs.color.expand(faces_k.shape + nurbs.color.shape))

        offset += verts_k.shape[0]

    verts_vector = torch.cat(verts_vector, dim=0)
    faces_vector = torch.cat(faces_vector, dim=0)
    colors_vector = torch.cat(colors_vector, dim=0)

    projected_verts = []
    cameras_rotation = []

    for camera in cameras:
        projected_verts.append(camera.transform(verts_vector))
        cameras_rotation.append(camera.extrinsics.R)

    projected_verts = torch.stack(projected_verts)
    cameras_rotation = torch.cat(cameras_rotation, dim=0)

    e1 = verts_vector[faces_vector[..., 1]] - verts_vector[faces_vector[..., 0]]
    e2 = verts_vector[faces_vector[..., 2]] - verts_vector[faces_vector[..., 0]]

    normals = torch.cross(e1, e2, dim=-1)
    normals = torch.nn.functional.normalize(normals, dim=-1)

    face_normals = (cameras_rotation @ normals.T).mT

    face_vertices_z = projected_verts[..., faces_vector, 2]
    face_vertices_image = projected_verts[..., faces_vector, :2]
    face_features = colors_vector.unsqueeze(0).expand(len(cameras), -1, -1, -1).contiguous()
    face_normals_z = face_normals[..., 2]

    rendered_features, rendered_soft_mask, rendered_face_idx = dibr_rasterization(H, W, face_vertices_z, face_vertices_image, face_features, face_normals_z, rast_backend=backend)

    return rendered_features

In [8]:
a = render(scene, cameras, H=height, W=width)
a.shape

torch.Size([2, 17444, 3])
torch.Size([2, 17444, 3, 2])
torch.Size([2, 17444, 3, 3])
torch.Size([2, 17444])


torch.Size([2, 1080, 1920, 3])

In [9]:
img = a[0].detach().cpu()
img = img.byte()
fig = px.imshow(img)
fig.show()

In [21]:
@torch.no_grad()
def visualize(
        scene: list[NURBS],
        height: int = 512,
        width: int = 512
) -> None:

    def render_visualize(camera: Camera) -> UInt8[torch.Tensor, "H W 3"]:
        face_normals = (camera.extrinsics.R @ normals.T).mT

        projected_verts = camera.transform(verts_vector)

        face_vertices_z = projected_verts[..., faces_vector, 2].unsqueeze(0)
        face_vertices_image = projected_verts[..., faces_vector, :2].unsqueeze(0)
        face_normals_z = face_normals[..., 2]

        rendered_features, rendered_soft_mask, rendered_face_idx = dibr_rasterization(height, width, face_vertices_z, face_vertices_image, face_features, face_normals_z, rast_backend=backend)

        return rendered_features[0].byte()

    verts_vector = []
    faces_vector = []
    colors_vector = []
    offset = 0

    for nurbs in scene:
        verts_k, faces_k = nurbs.tessellate()

        verts_vector.append(verts_k)
        faces_vector.append(faces_k + offset)
        colors_vector.append(nurbs.color.expand(faces_k.shape + nurbs.color.shape))

        offset += verts_k.shape[0]

    verts_vector = torch.cat(verts_vector, dim=0)
    faces_vector = torch.cat(faces_vector, dim=0)
    colors_vector = torch.cat(colors_vector, dim=0)


    e1 = verts_vector[faces_vector[..., 1]] - verts_vector[faces_vector[..., 0]]
    e2 = verts_vector[faces_vector[..., 2]] - verts_vector[faces_vector[..., 0]]

    normals = torch.cross(e1, e2, dim=-1)
    normals = torch.nn.functional.normalize(normals, dim=-1)

    face_features = colors_vector.unsqueeze(0).expand(1, -1, -1, -1).contiguous()

    turnable_visualizer = IpyTurntableVisualizer(
        height=height,
        width=width,
        camera=camera,
        render=render_visualize
    )

    turnable_visualizer.show()




In [22]:
visualize(scene)

Canvas(height=512, width=512)

Output()